In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [2]:
import base64
import io
import pandas as pd
from PIL import Image
import torchvision.transforms as transforms
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn import init

In [3]:
# Ensure every computation happens on the GPU when available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'

In [4]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [58]:
import zipfile
import os

# Path to the zip file
zip_file_path = '/content/drive/MyDrive/IMAGES_RESULTS_25.zip'

# Destination folder to extract the contents
destination_folder = '/content/image'

# Create the destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(destination_folder)

print(f"Files have been extracted to: {destination_folder}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IMAGES_RESULTS_25.zip'

In [59]:
image_paths = ['/content/image/000000182212.jpg',
               '/content/image/000000046267.jpg',
               '/content/image/000000020619.jpg',
               '/content/image/000000380338.jpg',
               '/content/image/000000544309.jpg',
               '/content/image/000000448856.jpg',
               '/content/image/000000425520.jpg',
               '/content/image/000000515809.jpg',
               '/content/image/000000353651.jpg',
               '/content/image/000000382669.jpg',
               '/content/image/000000579002.jpg',
               '/content/image/000000522365.jpg',
               '/content/image/000000272269.jpg',
               '/content/image/000000274337.jpg',
               '/content/image/000000428304.jpg',
               '/content/image/000000548661.jpg',
               '/content/image/000000266165.jpg',
               '/content/image/000000409092.jpg',
               '/content/image/000000225060.jpg',
               '/content/image/000000502326.jpg',
               '/content/image/000000329546.jpg',
               '/content/image/000000481120.jpg',
               '/content/image/000000464965.jpg',
               '/content/image/000000424604.jpg',
               '/content/image/000000512644.jpg']



short_descriptions = [
    "In a dark basement, there is a white",
    "In a shiny bathroom, the walls sparkle like",
    "There is a big table full of yummy",
    "Pink cakes and lollipops rest on white tables",
    "The cake is so colorful with chocolate and",
    "In a funny bathroom, there are two shiny",
    "In a happy green bathroom, there are funny",
    "The bathroom has a white toilet and a",
    "In a shiny bathroom, there is a big",
    "There's a man on a shiny, old motorcycle",
    "There's a big building with a clock inside",
    "The green bowl is on the table. It",
    "There is a big, yummy cake on a",
    "A big parade is happening! A police motorcycle",
    "A fluffy cat is on a table. It",
    "The orange kitty sits on the table beside",
    "The kitty is very funny. It stands in",
    "The cat is eating its food. It's funny",
    "A young man is sitting in a small",
    "The toilet has a big, round light above",
    "In a tiny bathroom, there is a white",
    "There are tiny green beads and nuts inside",
    "A man sits at his desk with a",
    "The bowl has yummy fruit like apples, bananas,",
    "In a big parking lot, two cool motorbikes"
]


In [12]:
# for the testing purposes only
df = pd.read_csv('/content/coco_30k_with_ShortDesc_Final_cleaned.csv')
print("Columns in dataframe is: ", df.columns)
df.rename(columns={'ShortDesc': 'caption'}, inplace=True)

Columns in dataframe is:  Index(['image_id', 'captions', 'ShortDesc'], dtype='object')


In [13]:
df.head()

,image_id,captions,caption
0,391895,['A man with a red helmet on a small moped on ...,A man in a red helmet zooms on his little bike...
1,522418,['A woman wearing a net on her head cutting a ...,A lady in a funny net on her head is slicing a...
2,184613,['A child holding a flowered umbrella and pett...,A little boy stands with a bright umbrella. He...
3,318219,['A young boy standing in front of a computer ...,A little boy sits in front of a glowing comput...
4,60623,['A young girl inhales with the intent of blow...,The girl takes a big breath and feels excited....


In [14]:
df.shape

(28931, 3)

In [15]:
#himanshu

# Extract all text from the 'caption' column
all_text = ''.join(df['caption'].fillna('').tolist())  # Fill NaNs with empty strings

# Extract unique characters from the 'caption' column
chars = sorted(list(set(all_text)))
chars.append('<pad>')
chars.append('<start>')

# Create mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Verify that all characters in all_text are mapped
for char in all_text:
    assert char in stoi, f"Character '{char}' is not in stoi!"

# Functions for encoding and decoding
encode = lambda s: [stoi[c] for c in s]  # Converts string to list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # Converts list of integers back to string


# Vocabulary size
vocab_size = len(stoi.keys())

# Debugging prints
print(f"Unique characters: {chars}")
print(f"stoi: {stoi}")
print(f"itos: {itos}")
print(f"Vocabulary size: {vocab_size}")


Unique characters: ['\n', ' ', '!', '"', '$', '&', "'", ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', 'é', 'ñ', '’', '“', '”', '🌊', '🌟', '📖', '🥳', '🧸', '🪁', '<pad>', '<start>']
stoi: {'\n': 0, ' ': 1, '!': 2, '"': 3, '$': 4, '&': 5, "'": 6, ',': 7, '-': 8, '.': 9, '/': 10, '0': 11, '1': 12, '2': 13, '3': 14, '4': 15, '5': 16, '6': 17, '7': 18, '8': 19, '9': 20, ':': 21, '<': 22, '>': 23, '?': 24, 'A': 25, 'B': 26, 'C': 27, 'D': 28, 'E': 29, 'F': 30, 'G': 31, 'H': 32, 'I': 33, 'J': 34, 'K': 35, 'L': 36, 'M': 37, 'N': 38, 'O': 39, 'P': 40, 'Q': 41, 'R': 42, 'S': 43, 'T': 44, 'U': 45, 'V': 46, 'W': 47, 'X': 48, 'Y': 49, 'Z': 50, '_': 51, 'a': 52, 'b': 53, 'c': 54, 'd': 55, 'e': 56

In [16]:
type(stoi['<start>'])

int

In [17]:
import torch
import torch.nn as nn

class PatchEmbeddings(nn.Module):
    def __init__(self, img_size=224, patch_size=16, hidden_dim=512):
        super().__init__()

        # Store the input image size
        self.img_size = img_size

        # Store the size of each patch
        self.patch_size = patch_size

        # Calculate the total number of patches
        self.num_patches = (img_size // patch_size) ** 2

        # First convolutional layer to extract patch embeddings
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=hidden_dim,
                               kernel_size=patch_size, stride=patch_size)

        # Add more layers to increase parameters
        self.conv2 = nn.Conv2d(in_channels=hidden_dim, out_channels=hidden_dim,
                               kernel_size=3, stride=1, padding=1)
        self.norm1 = nn.BatchNorm2d(hidden_dim)
        self.activation1 = nn.ReLU()



        # Optionally, add a final fully connected layer for further embedding refinement
        self.fc = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, X):
        # First convolutional layer
        X = self.conv1(X)

        # Additional convolutional layers with normalization and activation
        X = self.conv2(X)
        X = self.norm1(X)
        X = self.activation1(X)

        # X = self.conv3(X)
        # X = self.norm2(X)
        # X = self.activation2(X)

        # Flatten the spatial dimensions (height and width) of the patch embeddings
        X = X.flatten(2)

        # Transpose the dimensions to obtain the shape [batch_size, num_patches, hidden_dim]
        X = X.transpose(1, 2)

        # Apply the fully connected layer
        X = self.fc(X)

        return X


In [18]:
#testing
img_size, patch_size,  num_hiddens, batch_size = 224, 16, 512, 32
patch_embeddings = PatchEmbeddings(img_size, patch_size, num_hiddens )
X = torch.zeros(batch_size, 3, img_size, img_size)
patch_embeddings(X).shape

torch.Size([32, 196, 512])

In [19]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in patch_embeddings.parameters() if p.requires_grad)/1_000_000
# num_params_millions = num_params / 1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 3.017216


In [20]:
#swapping linear for lazy linear for simplicity. Lazylinear can accept any arbitrary input dimension without having it specified

class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, is_decoder=True):
        super().__init__()

        # Define the layers of the MLP
        layers = [
            # First linear layer that expands the input dimension from n_embd to 4 * n_embd
            nn.Linear(n_embd, 4 * n_embd),

            # Activation function: ReLU if is_decoder is True, else GELU
            nn.ReLU() if is_decoder else nn.GELU(),

            # Second linear layer that projects the intermediate dimension back to n_embd
            nn.Linear(4 * n_embd, n_embd),

            # Dropout layer for regularization
            nn.Dropout(dropout)
        ]

        # Create a sequential container to hold the layers
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # Pass the input through the MLP layers
        return self.net(x)


In [21]:
#For the sake of this example consider embedding size to be 128
n_embd = 128
testmlp = MLP(n_embd)
mlp_input = torch.zeros(batch_size, 3, n_embd)
testmlp_out = testmlp(mlp_input)
testmlp_out.shape

torch.Size([32, 3, 128])

In [22]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in testmlp.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 0.131712


In [23]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.1, is_decoder=False):
        super().__init__()

        # Linear layer for key projection
        self.key = nn.Linear(n_embd, head_size, bias=False)

        # Linear layer for query projection
        self.query = nn.Linear(n_embd, head_size, bias=False)

        # Linear layer for value projection
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Dropout layer for regularization
        self.dropout = nn.Dropout(dropout)

        # Flag indicating whether this head is used in the decoder
        self.is_decoder = is_decoder

    def forward(self, x):
        # Get the batch size (B), sequence length (T), and embedding dimension (C) from the input tensor
        B, T, C = x.shape

        # Compute key, query, and value projections
        k = self.key(x)   # Shape: [B, T, head_size]
        q = self.query(x) # Shape: [B, T, head_size]
        v = self.value(x) # Shape: [B, T, head_size]

        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # Shape: [B, T, T]


        if self.is_decoder:
            # If this head is used in the decoder, apply a causal mask to the attention scores
            # to prevent attending to future positions
            tril = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
            wei = wei.masked_fill(tril == 0, float('-inf'))

        # Apply softmax to the attention scores to obtain attention probabilities
        wei = F.softmax(wei, dim=-1) # Shape: [B, T, T]

        # Apply dropout to the attention probabilities for regularization
        wei = self.dropout(wei)

        # Perform weighted aggregation of values using the attention probabilities
        out = wei @ v # Shape: [B, T, head_size]

        return out


In [24]:
#Example values for testing
n_embd, head_size, batch_size = 128, 16, 32

testhead = Head(n_embd, head_size)
head_input = torch.zeros(batch_size, 3, n_embd)
testhead_out = testhead(head_input)
testhead_out.shape # (B, T,H_size)

torch.Size([32, 3, 16])

In [25]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in testhead.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 0.006144


In [26]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, num_heads, dropout=0.1, is_decoder=False):
        super().__init__()

        # Ensure that the embedding dimension is divisible by the number of heads
        assert n_embd % num_heads == 0, "n_embd must be divisible by num_heads"

        # Create a ModuleList of attention heads
        self.heads = nn.ModuleList([
            Head(n_embd, n_embd // num_heads, dropout, is_decoder)
            for _ in range(num_heads)
        ])

        # Linear layer for projecting the concatenated head outputs
        self.proj = nn.Linear(n_embd, n_embd)

        # Dropout layer for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Apply each attention head to the input tensor
        head_outputs = [h(x) for h in self.heads]

        # Concatenate the outputs from all heads along the last dimension
        out = torch.cat(head_outputs, dim=-1)

        # Apply the projection layer to the concatenated outputs
        out = self.proj(out)

        # Apply dropout to the projected outputs for regularization
        out = self.dropout(out)

        return out


In [27]:
#Example values for testing
n_embd, n_head = 128, 8
testmha = MultiHeadAttention(n_embd, n_head)
head_input = torch.zeros(batch_size, 3, n_embd)
testmha_out = testmha(head_input)
testmha_out.shape # (B, T,H_size*n_heads = n_embed)

torch.Size([32, 3, 128])

In [28]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in testmha.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 0.065664


In [29]:
class Block(nn.Module):
    def __init__(self, n_embd, num_heads, dropout=0.1, is_decoder=False):
        super().__init__()

        # Layer normalization for the input to the attention layer
        self.ln1 = nn.LayerNorm(n_embd)

        # Multi-head attention module
        self.attn = MultiHeadAttention(n_embd, num_heads, dropout, is_decoder)

        # Layer normalization for the input to the FFN
        self.ln2 = nn.LayerNorm(n_embd)

        # Feed-forward neural network (FFN)
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # Expand the dimension
            nn.GELU(),  # Activation function
            nn.Linear(4 * n_embd, n_embd),  # Project back to the original dimension
        )

    def forward(self, x):
        original_x = x  # Save the input for the residual connection

        # Apply layer normalization to the input
        x = self.ln1(x)

        # Apply multi-head attention
        attn_output = self.attn(x)

        # Add the residual connection (original input) to the attention output
        x = original_x + attn_output

        # Apply layer normalization to the input to the FFN
        x = self.ln2(x)

        # Apply the FFN
        ffn_output = self.ffn(x)

        # Add the residual connection (input to FFN) to the FFN output
        x = x + ffn_output

        return x

In [30]:
#Example values for testing
n_embd, head_size, batch_size = 128, 16, 32

testblock = Block(n_embd, n_head)
block_input = torch.zeros(batch_size, 3, n_embd)
testblock_out = testblock(block_input)
testblock_out.shape

torch.Size([32, 3, 128])

In [31]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in testblock.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 0.197888


Now all this can be be put together to implement a Vision Transformer

In [32]:
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout):
        super().__init__()

        # Patch embedding layer to convert the input image into patches
        self.patch_embedding = PatchEmbeddings(img_size, patch_size, num_hiddens)

        # Learnable classification token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))

        # Calculate the number of patches
        num_patches = (img_size // patch_size) ** 2

        # Learnable position embedding
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, num_hiddens))

        # Dropout layer for the embeddings
        self.dropout = nn.Dropout(emb_dropout)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([Block(num_hiddens, num_heads, blk_dropout, is_decoder=False) for _ in range(num_blks)])

        # Layer normalization for the final representation
        self.layer_norm = nn.LayerNorm(num_hiddens)

    def forward(self, X):
        # Convert the input image into patch embeddings
        x = self.patch_embedding(X)

        # Expand the classification token to match the batch size
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)

        # Concatenate the classification token with the patch embeddings
        x = torch.cat((cls_tokens, x), dim=1)

        # Add the position embedding to the patch embeddings
        x += self.pos_embedding

        # Apply dropout to the embeddings
        x = self.dropout(x)

        # Pass the embeddings through the transformer blocks
        for block in self.blocks:
            x = block(x)

        # Apply layer normalization to the final representation
        x = self.layer_norm(x[:, 0])

        return x


In [33]:
#For purposes of testing
img_size, patch_size, num_hiddens, n_head, num_blks, dropout = 224, 16, 512, 8, 3, 0.1

testvit = ViT(img_size, patch_size, num_hiddens, n_head, num_blks, dropout, dropout)
vit_input = torch.zeros(batch_size, 3, img_size, img_size)
testvit_out = testvit(vit_input)
testvit_out.shape

torch.Size([32, 512])

In [34]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in testvit.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 12.57216


In [35]:
class MultiModalProjector(nn.Module):
    def __init__(self, n_embd, image_embed_dim, dropout=0.1):
        super().__init__()

        # Define the projection network
        self.net = nn.Sequential(
            # Linear layer to expand the image embedding dimension
            nn.Linear(image_embed_dim, 4 * image_embed_dim),

            # GELU activation function
            nn.GELU(),

            # Linear layer to project the expanded image embeddings to the text embedding dimension
            nn.Linear(4 * image_embed_dim, n_embd),

            # Dropout layer for regularization
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # Pass the input through the projection network
        x = self.net(x)
        return x


In [36]:
#Example values for testing
n_embd,num_hiddens = 128, 512

testmmp = MultiModalProjector(n_embd,num_hiddens)
mmp_input = testvit_out
testmmp_out = testmmp(mmp_input)
testmmp_out.shape

torch.Size([32, 128])

In [37]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in testmmp.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 1.312896


In [38]:
class DecoderLanguageModel(nn.Module):
    def __init__(self, n_embd, image_embed_dim, vocab_size, num_heads, n_layer, use_images=False, max_seq_len=5000):
        super().__init__()

        self.use_images = use_images
        self.max_seq_len = max_seq_len # Store max_seq_len for later use


        # Token embedding table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # Position embedding table
        self.position_embedding_table = nn.Embedding(max_seq_len, n_embd) # previously 1000

        if use_images:
            # Image projection layer to align image embeddings with text embeddings
            self.image_projection = MultiModalProjector(n_embd, image_embed_dim)

        # Stack of transformer decoder blocks
        self.blocks = nn.Sequential(*[Block(n_embd, num_heads, is_decoder=True) for _ in range(n_layer)])

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Language modeling head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, image_embeds=None, targets=None):
        # Get token embeddings from the input indices
        tok_emb = self.token_embedding_table(idx)

        if self.use_images and image_embeds is not None:
            # Project and concatenate image embeddings with token embeddings
            img_emb = self.image_projection(image_embeds).unsqueeze(1)
            tok_emb = torch.cat([img_emb, tok_emb], dim=1)

        # Get position embeddings, ensuring sequence length doesn't exceed the maximum
        seq_len = tok_emb.size(1)
        pos_emb = self.position_embedding_table(torch.arange(min(seq_len, self.max_seq_len), device=device).long()).unsqueeze(0) # clamped sequence length and added .long() for type safety

        # Add position embeddings to token embeddings
        x = tok_emb + pos_emb

        # Pass through the transformer decoder blocks
        x = self.blocks(x)

        # Apply final layer normalization
        x = self.ln_f(x)

        # Get the logits from the language modeling head
        logits = self.lm_head(x)

        if targets is not None:
            if self.use_images and image_embeds is not None:
                # Prepare targets by concatenating a dummy target for the image embedding
                batch_size = idx.size(0)
                targets = torch.cat([torch.full((batch_size, 1), -100, dtype=torch.long, device=device), targets], dim=1)

            # Compute the cross-entropy loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
            return logits, loss

        return logits


    def generate(self, idx, image_embeds, max_new_tokens):
        # Get the batch size and sequence length
        B, T = idx.shape

        # Initialize the generated sequence with the input indices
        generated = idx

        if self.use_images and image_embeds is not None:
            # Project and concatenate image embeddings with token embeddings
            img_emb = self.image_projection(image_embeds).unsqueeze(1)
            current_output = torch.cat([img_emb, self.token_embedding_table(idx)], dim=1)
        else:
            current_output = self.token_embedding_table(idx)

        # Generate new tokens iteratively
        for i in range(max_new_tokens):
            # Get the current sequence length
            T_current = current_output.size(1)

            # Get position embeddings for the current sequence length
            current_pos_emb = self.position_embedding_table(torch.arange(T_current, device=device)).unsqueeze(0)

            # Add position embeddings to the current output
            current_output += current_pos_emb

            # Pass through the transformer decoder blocks
            for block in self.blocks:
                current_output = block(current_output)

            # Get the logits for the last token
            logits = self.lm_head(current_output[:, -1, :])

            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token based on the probabilities
            idx_next = torch.multinomial(probs, num_samples=1)

            # Concatenate the generated token to the generated sequence
            generated = torch.cat((generated, idx_next), dim=1)

            # Get the embeddings for the generated token
            idx_next_emb = self.token_embedding_table(idx_next)

            # Concatenate the generated token embeddings to the current output
            current_output = torch.cat((current_output, idx_next_emb), dim=1)

        return generated

In [39]:
# I use n_layer to represent number of decoder transformer blocks and n_blks for the vision encoder to avoid confusion
model = DecoderLanguageModel(n_embd=128, image_embed_dim=512, vocab_size=1000, num_heads=8, n_layer=6, use_images=True)
model.to(device)
# Dummy input
B, T = 10, 50
idx = torch.randint(0, 1000, (B, T)).to(device)
image_embeds = torch.randn(B, 512).to(device)  # Assume image_embed_dim is 256

targets = torch.randint(0, vocab_size, (B, T)).to(device)  # Only if you want to compute loss

# Test forward pass
# Check if you need to calculate loss by providing targets
if targets is not None:
    logits, loss = model(idx, image_embeds, targets)
    print(f"Logits shape: {logits.shape}, Loss: {loss}")
else:
    logits = model(idx, image_embeds)  # Call without targets
    print(f"Logits shape: {logits.shape}")

# Test generation
generated = model.generate(idx, image_embeds, max_new_tokens=20)
print(f"Generated sequence shape: {generated.shape}")



Logits shape: torch.Size([10, 51, 1000]), Loss: 7.071423530578613
Generated sequence shape: torch.Size([10, 70])


In [40]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 3.39748


In [41]:
class VisionLanguageModel(nn.Module):
    def __init__(self, n_embd, image_embed_dim, vocab_size, n_layer, img_size, patch_size, num_heads, num_blks, emb_dropout, blk_dropout):
        super().__init__()

        # Set num_hiddens equal to image_embed_dim
        num_hiddens = image_embed_dim

        # Assert that num_hiddens is divisible by num_heads
        assert num_hiddens % num_heads == 0, "num_hiddens must be divisible by num_heads"

        # Initialize the vision encoder (ViT)
        self.vision_encoder = ViT(img_size, patch_size, num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout)

        # Initialize the language model decoder (DecoderLanguageModel)
        self.decoder = DecoderLanguageModel(n_embd, image_embed_dim, vocab_size, num_heads, n_layer, use_images=True)

    def forward(self, img_array, idx, targets=None):
        # Get the image embeddings from the vision encoder
        image_embeds = self.vision_encoder(img_array)

        # Check if the image embeddings are valid
        if image_embeds.nelement() == 0 or image_embeds.shape[1] == 0:
            raise ValueError("Something is wrong with the ViT model. It's returning an empty tensor or the embedding dimension is empty.")

        if targets is not None:
            # If targets are provided, compute the logits and loss
            logits, loss = self.decoder(idx, image_embeds, targets)
            return logits, loss
        else:
            # If targets are not provided, compute only the logits
            logits = self.decoder(idx, image_embeds)
            return logits

    def generate(self, img_array, idx, max_new_tokens):
        # Get the image embeddings from the vision encoder
        image_embeds = self.vision_encoder(img_array)

        # Check if the image embeddings are valid
        if image_embeds.nelement() == 0 or image_embeds.shape[1] == 0:
            raise ValueError("Something is wrong with the ViT model. It's returning an empty tensor or the embedding dimension is empty.")

        # Generate new tokens using the language model decoder
        generated_tokens = self.decoder.generate(idx, image_embeds, max_new_tokens)
        return generated_tokens

In [42]:
image_embed_dim = num_hiddens

In [43]:
n_layer, block_size =  6, 32

# Initialize the model
model = VisionLanguageModel(n_embd, image_embed_dim, vocab_size,  n_layer, img_size, patch_size, n_head, num_blks, dropout, dropout)
model.to(device)

# Create dummy data with correct dimensions
dummy_img = torch.randn(1, 3, img_size, img_size).to(device)  # Correct shape for image input
dummy_idx = torch.randint(0, vocab_size, (1, block_size)).to(device)  # Correct shape for text input

# Forward pass to initialize all parameters
try:
    output = model(dummy_img, dummy_idx)  # Output for debugging
    print("Output from initialization forward pass:", output)
except RuntimeError as e:
    print(f"Runtime Error during forward pass: {str(e)}")
    print("Check layer configurations and input shapes.")

Output from initialization forward pass: tensor([[[ 0.7038, -0.4214,  0.1247,  ..., -0.1658,  0.0638, -0.3665],
         [ 0.0096, -0.6768,  0.6301,  ...,  0.0149, -0.1284, -0.4467],
         [ 0.6901, -0.6334,  0.3313,  ...,  0.4737, -0.0072,  0.3392],
         ...,
         [-0.1683,  0.3327, -0.3809,  ..., -0.4557, -1.2834, -0.4058],
         [ 0.4081, -1.6672,  0.1290,  ...,  0.7339, -0.4530, -0.6578],
         [ 0.2949, -1.4786,  0.6094,  ...,  1.2000, -0.6599, -0.2450]]],
       device='cuda:0', grad_fn=<ViewBackward0>)


In [44]:
from PIL import Image
from torchvision import transforms

def jpg_to_tensor(file_path, img_size=224):

    # Open the image file
    pil_image = Image.open(file_path)

    # Ensure the image is in RGB format
    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')
    # print("SHAPE of rgb image is ", pil_image.size)
    # Define preprocessing transforms
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),  # Resize to desired dimensions
        transforms.ToTensor(),                   # Convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
    ])

    # Apply transforms and return the tensor
    # print("SHAPE of tensor is ", transform(pil_image).shape)
    return transform(pil_image).unsqueeze(0)


In [45]:
#Adjusting the data loader from makemore for multimodal data
def get_batch(df, batch_size, split='train', img_size=224, val_batch_size = 8):
    # Split data into training and validation sets
    n = int(0.9 * len(df))  # first 90% will be train, rest val
    df_train = df.iloc[:n]
    df_val = df.iloc[n:]
    data = df_train if split == 'train' else df_val
    batch_size = batch_size if split == 'train' else val_batch_size
    replace = False if split == 'train' else True
    batch = data.sample(n=batch_size, replace=replace)

    images = torch.cat([jpg_to_tensor(img, img_size) for img in batch['image']], dim=0).to(device)
    # print("shape of image is ", images.shape)
    text_indices = [torch.tensor(encode(desc), dtype=torch.long) for desc in batch['caption']]
    text_indices = [torch.cat([torch.tensor([stoi['<start>']], dtype=torch.long), tensor]) for tensor in text_indices]
    max_length = max(len(t) for t in text_indices)

    padded_text = torch.full((batch_size, max_length), fill_value=stoi['<pad>'], dtype=torch.long).to(device)
    for i, text in enumerate(text_indices):
        padded_text[i, :len(text)] = text

    targets = torch.cat([padded_text[:, 1:], torch.full((batch_size, 1), fill_value=stoi['<pad>'], dtype=torch.long, device=device)], dim=1).to(device)

    # Truncate or pad targets to match the length of padded_text
    if targets.size(1) > padded_text.size(1):
        targets = targets[:, :padded_text.size(1)]
    elif targets.size(1) < padded_text.size(1):
        targets = torch.cat([targets, torch.full((batch_size, padded_text.size(1) - targets.size(1)), fill_value=stoi['<pad>'], dtype=torch.long, device=device)], dim=1)

    return images, padded_text, targets

In [46]:
def estimate_loss(model, df, split, img_size=224, val_batch_size = 8):
    losses = []
    model.eval()
    with torch.no_grad():
      for _ in range(eval_iters):
          images, idx, targets = get_batch(df, batch_size, split, img_size, val_batch_size=val_batch_size)
          _, loss = model(images, idx, targets)
          losses.append(loss.item())
    return sum(losses) / len(losses)

In [47]:
import matplotlib.pyplot as plt

def plot_training_loss(train_losses, epochs):
    """Plots the training loss per epoch."""
    plt.plot(range(1, epochs + 1), train_losses, label="Training Loss", marker='o', color='blue')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training Loss per Epoch")
    plt.legend()
    plt.grid()
    plt.show()


def plot_validation_loss(val_losses, epochs):
    """Plots the validation loss per epoch."""
    plt.plot(range(1, epochs + 1), val_losses, label="Validation Loss", marker='o', color='orange')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Validation Loss per Epoch")
    plt.legend()
    plt.grid()
    plt.show()


In [48]:
train_losses = []  # List to store average training losses per epoch
val_losses = []

def train_model(model, df, epochs, vocab_size, img_size=224, save_path="./latest_model.pth"):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.to(device)

    # Initialize previous validation losses
    prev_val_loss = float('inf')  # Validation loss for the last epoch
    second_prev_val_loss = float('inf')  # Validation loss for the second-to-last epoch

    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0  # To calculate average training loss
        for iteration in range(max_iters):
            images, idx, targets = get_batch(df, batch_size, 'train', img_size)
            optimizer.zero_grad()
            logits, loss = model(images, idx, targets)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

            # Decode and print input and model's output sentences
            input_sentences = [decode(seq.tolist()).replace('<pad>', '') for seq in idx]
            model_output_indices = logits.argmax(dim=-1)  # Get the most probable tokens from logits
            output_sentences = [decode(seq.tolist()).replace('<pad>', '') for seq in model_output_indices]

            for i, (inp, out) in enumerate(zip(input_sentences, output_sentences)):
                print(f"Epoch: {epoch + 1}, Iteration: {iteration + 1}, Datapoint: {i + 1}")
                print(f"Input Sentence: {inp}")
                print(f"Model Output Sentence: {out}")
                print("-" * 50)

        # Calculate average training loss for the epoch
        avg_train_loss = epoch_train_loss / max_iters
        train_losses.append(avg_train_loss)  # Store average training loss
        print(f"Average Training Loss after epoch {epoch + 1}: {avg_train_loss}")

        # Validation loss
        val_loss = estimate_loss(model, df, 'val', img_size, val_batch_size=8)
        val_losses.append(val_loss)  # Store validation loss
        print(f"Validation Loss For epoch {epoch + 1}: {val_loss}")

        # Save the model checkpoint (overwrite previous one)
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': val_loss
        }, save_path)
        print(f"Model checkpoint saved/updated at {save_path}")

        # Update previous validation losses
        second_prev_val_loss = prev_val_loss
        prev_val_loss = val_loss

    print("Training losses are ", train_losses)
    print("Validation losses are ", val_losses)


In [94]:
# # this for 5.65 m model
# batch_size = 128 # how many independent sequences will we process in parallel?
# val_batch_size = 8
# block_size = 32 # what is the maximum context length for predictions?
# max_iters = 200
# eval_interval = 50
# learning_rate = 1e-3
# epochs= 40
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# eval_iters = 375
# num_blks= 1 #3
# head_size = 12 #16
# n_embd =  96 #256 #128
# n_head = 8 #16 #8
# n_layer = 4 #10 #8
# dropout = 0.1
# img_size = 224
# patch_size = 16
# image_embed_dim = 400
# emb_dropout = blk_dropout = 0.1

# # this is for 16M model
# batch_size = 64 # how many independent sequences will we process in parallel?
# val_batch_size = 8
# block_size = 32 # what is the maximum context length for predictions?
# max_iters = 400
# eval_interval = 50
# learning_rate = 1e-3
# epochs= 60
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# eval_iters = 375
# num_blks= 3
# head_size = 16
# n_embd = 128
# n_head = 8
# n_layer = 8
# dropout = 0.1
# img_size = 224
# patch_size = 16
# image_embed_dim = 512
# emb_dropout = blk_dropout = 0.1


## ye 25M wale ke hyperparamters hain
batch_size = 128 # how many independent sequences will we process in parallel?
val_batch_size = 8
block_size = 32 # what is the maximum context length for predictions?
max_iters = 200
eval_interval = 50
learning_rate = 1e-3
epochs= 40
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 375
num_blks= 5 #3
head_size = 12 #16
n_embd =  192 #256 #128
n_head = 16 #16 #8
n_layer = 10 #10 #8
dropout = 0.1
img_size = 224
patch_size = 16
image_embed_dim = 512
emb_dropout = blk_dropout = 0.1

Let's train!! Optional: Use MLFlow to track eval loss as the model is being trained

In [95]:
# Initialize the model
model = VisionLanguageModel(n_embd, image_embed_dim, vocab_size, n_layer, img_size, patch_size, n_head, num_blks, emb_dropout, blk_dropout)
model.to(device)

# Dummy data to initialize lazy modules
dummy_img = torch.randn(1, 3, img_size, img_size).to(device)
dummy_idx = torch.randint(0, vocab_size, (1, block_size)).to(device)
model(dummy_img, dummy_idx)  # Forward pass to initialize all parameters

# Train the model


tensor([[[-0.6881,  0.7415,  0.5470,  ..., -1.5379,  0.5576, -0.0292],
         [-0.7438,  0.1390,  0.2314,  ..., -0.7415, -0.4362,  0.5923],
         [-0.9064,  0.9313,  0.6034,  ..., -0.8331,  0.7051,  0.0800],
         ...,
         [-1.0514, -0.5484,  0.5657,  ..., -1.1241, -0.1256,  0.9398],
         [-0.4982,  0.3935,  0.2039,  ..., -1.1385,  0.4868,  0.7402],
         [ 0.5900,  0.5737, -1.1066,  ..., -0.0695,  0.9872, -0.6729]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [96]:
import torch

# Assuming model is your VLM (Vision-Language Model) defined as a class
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)/1_000_000

print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 25.756572


In [97]:
# train_model(model, df, epochs, vocab_size, img_size)

# for testing purposes only

In [98]:

#load the model
# Load the checkpoint
# checkpoint_path = "/content/NanoVLMmini_ShortDesc_5dot65M.pth"  # for 5M model
# checkpoint_path = "/content/NanoVLMbase_ShortDesc_16M.pth"  # for 16M model
checkpoint_path = "/content/NanoVLMlarge_ShortDesc_25M.pth"  # for 25M model

checkpoint = torch.load(checkpoint_path, map_location=device)


# Load the model's state_dict
model.load_state_dict(checkpoint['model_state_dict'])

# Set model to evaluation mode for inference
model.eval()

print("Model successfully loaded and ready for inference.")

<ipython-input-98-d6bf4468cab4>:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


Model successfully loaded and ready for inference.


In [92]:
# isme image+text se inferencing ki hui hae

# import torch
# from PIL import Image
# from torchvision import transforms
# import zipfile
# import os
# import pandas as pd  # Import pandas for dataframe
# from google.colab import files  # For downloading files in Google Colab

# # Function to convert an image file path or PIL image to a tensor
# def image_to_tensor(image, img_size=224):
#     """
#     Convert an image file path or a PIL image object to a preprocessed tensor.
#     """
#     if isinstance(image, str):
#         image = Image.open(image).convert("RGB")  # Ensure it's in RGB mode

#     transform = transforms.Compose([
#         transforms.Resize((img_size, img_size)),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
#     ])

#     return transform(image).unsqueeze(0)  # Add batch dimension

# # Function to generate a story from an image and partial text
# def test_story_completion_with_image(model, partial_story, img, max_length=100, img_size=224):
#     model.eval()

#     partial_story = "<start> " + partial_story
#     encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
#     generated_indices = encoded_partial
#     img_tensor = image_to_tensor(img, img_size).to(device)

#     for _ in range(max_length - len(partial_story)):
#         logits = model(img_tensor, generated_indices, None)
#         next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Most probable next token
#         generated_indices = torch.cat([generated_indices, next_token], dim=1)

#         if next_token.item() == stoi['<pad>']:
#             break

#     completed_story = decode(generated_indices[0].tolist())
#     return completed_story

# # Function to run story completion for all images and descriptions
# def run_story_completion_batch(model, image_paths, short_descriptions):
#     completed_stories = []

#     for image_path, partial_story in zip(image_paths, short_descriptions):
#         print(f"Processing image: {image_path}")
#         print(f"Partial Story: {partial_story}")

#         # Generate the completed story
#         completed_story = test_story_completion_with_image(model, partial_story, image_path, max_length=1000)

#         print(f"Completed Story:\n{completed_story}\n")
#         completed_stories.append((partial_story, completed_story))

#     return completed_stories

# # Extract the zip file
# def extract_zip(zip_file_path, destination_folder):
#     os.makedirs(destination_folder, exist_ok=True)

#     with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#         zip_ref.extractall(destination_folder)

#     print(f"Files have been extracted to: {destination_folder}")

# # Example usage
# zip_file_path = '/content/IMAGES_RESULTS_25.zip'  # Path to your zip file
# destination_folder = '/content/image'  # Folder to extract images

# # Extract the zip file
# extract_zip(zip_file_path, destination_folder)

# # List of image paths and partial descriptions
# image_paths = ['/content/image/000000182212.jpg',
#                '/content/image/000000046267.jpg',
#                '/content/image/000000020619.jpg',
#                '/content/image/000000380338.jpg',
#                '/content/image/000000544309.jpg',
#                '/content/image/000000448856.jpg',
#                '/content/image/000000425520.jpg',
#                '/content/image/000000515809.jpg',
#                '/content/image/000000353651.jpg',
#                '/content/image/000000382669.jpg',
#                '/content/image/000000579002.jpg',
#                '/content/image/000000522365.jpg',
#                '/content/image/000000272269.jpg',
#                '/content/image/000000274337.jpg',
#                '/content/image/000000428304.jpg',
#                '/content/image/000000548661.jpg',
#                '/content/image/000000266165.jpg',
#                '/content/image/000000409092.jpg',
#                '/content/image/000000225060.jpg',
#                '/content/image/000000502326.jpg',
#                '/content/image/000000329546.jpg',
#                '/content/image/000000481120.jpg',
#                '/content/image/000000464965.jpg',
#                '/content/image/000000424604.jpg',
#                '/content/image/000000512644.jpg']

# short_descriptions = [
#     "In a dark basement, there is a white",
#     "In a shiny bathroom, the walls sparkle like",
#     "There is a big table full of yummy",
#     "Pink cakes and lollipops rest on white tables",
#     "The cake is so colorful with chocolate and",
#     "In a funny bathroom, there are two shiny",
#     "In a happy green bathroom, there are funny",
#     "The bathroom has a white toilet and a",
#     "In a shiny bathroom, there is a big",
#     "There's a man on a shiny, old motorcycle",
#     "There's a big building with a clock inside",
#     "The green bowl is on the table. It",
#     "There is a big, yummy cake on a",
#     "A big parade is happening! A police motorcycle",
#     "A fluffy cat is on a table. It",
#     "The orange kitty sits on the table beside",
#     "The kitty is very funny. It stands in",
#     "The cat is eating its food. It's funny",
#     "A young man is sitting in a small",
#     "The toilet has a big, round light above",
#     "In a tiny bathroom, there is a white",
#     "There are tiny green beads and nuts inside",
#     "A man sits at his desk with a",
#     "The bowl has yummy fruit like apples, bananas,",
#     "In a big parking lot, two cool motorbikes"
# ]

# # Run the batch processing
# completed_stories = run_story_completion_batch(model, image_paths, short_descriptions)

# # Convert the results into a dataframe
# df = pd.DataFrame(completed_stories, columns=['Partial Story', 'Completed Story'])

# # Save the dataframe to a CSV file
# csv_file_path = '/content/ShortDesc_16M_results.csv'
# df.to_csv(csv_file_path, index=False)

# # Provide a download link for the CSV file in Google Colab
# files.download(csv_file_path)


In [99]:
#isme only text se inferencing ki hui hae

import pandas as pd
import torch
from google.colab import files

def test_story_completion_batch(model, partial_stories, imgs=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model for multiple data points.
    """
    model.eval()
    completed_stories = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i, partial_story in enumerate(partial_stories):
        # Encode the partial story
        encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
        generated_indices = encoded_partial

        # Prepare the image tensor
        if imgs is not None and imgs[i] is not None:
            img_tensor = jpg_to_tensor(imgs[i], img_size).unsqueeze(0).to(device)
        else:
            img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

        # Generate the story
        for _ in range(max_length - len(partial_story)):
            logits = model(img_tensor, generated_indices, None)
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
            generated_indices = torch.cat([generated_indices, next_token], dim=1)

            if next_token.item() == stoi['<pad>']:
                break

        completed_story = decode(generated_indices[0].tolist())
        completed_stories.append(completed_story)

    return completed_stories

def run_story_completion_test_batch(model):
    """Run the story completion test and save results to CSV."""
    partial_stories = [
    "There is a man wearing a bright red helmet",
    "In a bright kitchen, a lady with a",
    "There is a happy boy with a bright",
    "There is a little boy at school sitting",
    "In a big room, there are many computers",
    "The lady is in the kitchen, and it",
    "The little girl sits at the table, her",
    "In a shiny kitchen, there is a big",
    "In a big, shiny kitchen, two men with",
    "In a busy kitchen, two chefs are working",
    "In a dark room, there are lots of",
    "In the kitchen, there are shiny black machines",
    "There’s a big kitchen with shiny sinks and",
    "The man is pedaling his bike fast, and",
    "The kitchen is long and skinny, like a",
    "In a big, sunny room, there is a",
    "In a bright kitchen, a young woman smiles",
    "The kitchen is big and a little messy,",
    "Once upon a time, a boy was riding",
    "In a cozy kitchen, there are shiny black",
    "There is a big red truck driving down",
    "In a cozy kitchen, there is a tall",
    "There are a bunch of wooden spoons on",
    "The boat is so big, and it has",
    "There are many kids at their desks using"
  ]

    completed_stories = test_story_completion_batch(model, partial_stories, imgs=None, max_length=100)

    df = pd.DataFrame({'Partial Story': partial_stories, 'Model Generated Story': completed_stories})
    csv_file_path = '/content/Rougescore_shortdesc_25M.csv'
    df.to_csv(csv_file_path, index=False)
    files.download(csv_file_path)

    return df

# Run the test
df = run_story_completion_test_batch(model)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
import torch
from PIL import Image
from torchvision import transforms

# Function to convert an image file path or PIL image to a tensor
def image_to_tensor(image, img_size=224):
    """
    Convert an image file path or a PIL image object to a preprocessed tensor.
    """
    # If input is a file path, open the image
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")  # Ensure it's in RGB mode

    # Define preprocessing transforms
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),  # Resize to desired dimensions
        transforms.ToTensor(),                   # Convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
    ])

    return transform(image).unsqueeze(0)  # Convert to tensor and add batch dimension

# Function to generate a story from an image and partial text
def test_story_completion_with_image(model, partial_story, img, max_length=100, img_size=224):
    """
    Generate a completed story using both an input image and a partial story.
    """
    model.eval()

    # Add <start> token at the beginning of the partial story
    partial_story = "<start> " + partial_story

    # Encode the partial story
    encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
    generated_indices = encoded_partial

    # Ensure `img` is correctly processed
    img_tensor = image_to_tensor(img, img_size).to(device)

    # Generate the story
    for _ in range(max_length - len(partial_story)):
        logits = model(img_tensor, generated_indices, None)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get most probable next token

        # Append the next token to the generated indices
        generated_indices = torch.cat([generated_indices, next_token], dim=1)

        # Stop if end-of-sequence or padding token is generated
        if next_token.item() == stoi['<pad>']:
            break

    # Decode the generated story
    completed_story = decode(generated_indices[0].tolist())
    return completed_story


# Function to test story completion with an image
def run_story_completion_with_image_test(model, image_path, partial_story):
    test_image = Image.open(image_path).convert("RGB")

    print(f"Partial Story: {partial_story}")
    completed_story = test_story_completion_with_image(model, partial_story, test_image, max_length=1000)

    print(f"Completed Story:\n{completed_story}\n")

# Example usage
image_path = "/content/000000020619.jpg"  # Replace with your image path
partial_story = "There is a big table full of yummy"
run_story_completion_with_image_test(model, image_path, partial_story)


Partial Story: There is a big table full of yummy
Completed Story:
<start> There is a big table full of yummy food! There are colorful fruits and vegetables on top. It looks like a fun party for my tummy!<pad>



In [68]:
import torch
from PIL import Image
from torchvision import transforms

# Function to convert an image file path or PIL image to a tensor
def image_to_tensor(image, img_size=224):
    """
    Convert an image file path or a PIL image object to a preprocessed tensor.
    """
    # If input is a file path, open the image
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")  # Ensure it's in RGB mode

    # Define preprocessing transforms
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),  # Resize to desired dimensions
        transforms.ToTensor(),                   # Convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
    ])

    return transform(image).unsqueeze(0)  # Convert to tensor and add batch dimension

# Function to generate a story from an image and partial text
def test_story_completion_with_image(model, partial_story, img, max_length=100, img_size=224):
    """
    Generate a completed story using both an input image and a partial story.
    """
    model.eval()

    # Add <start> token at the beginning of the partial story
    partial_story = "<start> " + partial_story

    # Encode the partial story
    encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
    generated_indices = encoded_partial

    # Ensure `img` is correctly processed
    img_tensor = image_to_tensor(img, img_size).to(device)

    # Generate the story
    for _ in range(max_length - len(partial_story)):
        logits = model(img_tensor, generated_indices, None)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get most probable next token

        # Append the next token to the generated indices
        generated_indices = torch.cat([generated_indices, next_token], dim=1)

        # Stop if end-of-sequence or padding token is generated
        if next_token.item() == stoi['<pad>']:
            break

    # Decode the generated story
    completed_story = decode(generated_indices[0].tolist())
    return completed_story


# Function to test story completion with an image
def run_story_completion_with_image_test(model, image_path, partial_story):
    test_image = Image.open(image_path).convert("RGB")

    print(f"Partial Story: {partial_story}")
    completed_story = test_story_completion_with_image(model, partial_story, test_image, max_length=1000)

    print(f"Completed Story:\n{completed_story}\n")

# Example usage
image_path = "/content/000000225060.jpg"  # Replace with your image path
partial_story = "A young man is sitting in a small "
run_story_completion_with_image_test(model, image_path, partial_story)


Partial Story: A young man is sitting in a small 
Completed Story:
<start> A young man is sitting in a small chair. He has a big box of shiny green bags and a picture. He looks happy and ready for a fun time!<pad>



In [85]:
def test_story_completion(model, partial_story, img=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model.

    Parameters:
    - model: The trained model.
    - partial_story: The input text to start the story completion.
    - img: Optional image tensor corresponding to the story. Provide a dummy image if None.
    - max_length: Maximum length of the completed story.
    - img_size: Size to which the image should be resized.
    """
    model.eval()

    # Encode the partial story
    encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
    generated_indices = encoded_partial

    # Prepare the image tensor, use a dummy image if img is None
    if img is not None:
        img_tensor = jpg_to_tensor(img, img_size).unsqueeze(0).to(device)
    else:
        # Create a dummy image tensor (e.g., all zeros with appropriate shape)
        img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

    # Generate the story
    for _ in range(max_length - len(partial_story)):
        # Forward pass through the model
        logits = model(img_tensor, generated_indices, None)  # Model's forward pass
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get the most likely next token

        # Append the next token to the generated indices
        generated_indices = torch.cat([generated_indices, next_token], dim=1)

        # Stop generating if a padding character or an unwanted token appears
        if next_token.item() == stoi['<pad>']:
            break

    # Decode the generated story
    completed_story = decode(generated_indices[0].tolist())
    return completed_story


# Function to test the story completion
def run_story_completion_test(model):
    partial_story = "Oh my goodness! Look at this super cozy room! It’s like a big hug made of furniture! "
    print(f"Partial Story: {partial_story}")

    completed_story = test_story_completion(model, partial_story, max_length=1000)
    print(f"Completed Story: {completed_story}\n")

# Run the test
run_story_completion_test(model)

Partial Story: Oh my goodness! Look at this super cozy room! It’s like a big hug made of furniture! 
Completed Story: Oh my goodness! Look at this super cozy room! It’s like a big hug made of furniture! The cat looks so cozy and fun!<pad>



In [ ]:
def test_story_completion_batch(model, partial_stories, imgs=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model for multiple datapoints.

    Parameters:
    - model: The trained model.
    - partial_stories: A list of input texts to start the story completion.
    - imgs: A list of image file paths or None for each story. Provide a dummy image if None.
    - max_length: Maximum length of the completed story.
    - img_size: Size to which the image should be resized.
    """
    model.eval()

    completed_stories = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Iterate through the batch of stories
    for i, partial_story in enumerate(partial_stories):
        # Encode the partial story
        encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
        generated_indices = encoded_partial

        # Prepare the image tensor
        if imgs is not None and imgs[i] is not None:
            img_tensor = jpg_to_tensor(imgs[i], img_size).unsqueeze(0).to(device)
        else:
            # Create a dummy image tensor (e.g., all zeros with appropriate shape)
            img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

        # Generate the story
        for _ in range(max_length - len(partial_story)):
            # Forward pass through the model
            logits = model(img_tensor, generated_indices, None)  # Model's forward pass
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get the most likely next token

            # Append the next token to the generated indices
            generated_indices = torch.cat([generated_indices, next_token], dim=1)

            # Stop generating if a padding character or an unwanted token appears
            if next_token.item() == stoi['<pad>']:
                break

        # Decode the generated story
        completed_story = decode(generated_indices[0].tolist())
        completed_stories.append(completed_story)

    return completed_stories


# Function to test story completion with multiple datapoints
def run_story_completion_test_batch(model):
    # Example batch of partial stories
    partial_stories = [
        "I can’t believe my eyes! This place looks magical, like something out of a fairytale! ",
        "The room was dimly lit, yet every corner sparkled with a mysterious glow. ",
        "As I stepped inside, the aroma of fresh flowers overwhelmed my senses. ",
        "The walls were lined with bookshelves, each filled with volumes that seemed ancient and wise. ",
        "Through the open window, I could hear the distant sound of waves crashing on the shore. ",
        "The fireplace crackled gently, casting a warm light across the room. ",
        "Outside, the snow fell silently, covering the world in a blanket of white. ",
        "The garden beyond the glass doors looked like a dream, alive with colors and fragrance. ",
        "In the center of the room, a grand piano stood, inviting me to play a tune. "
    ]

    # Example: No images provided, use dummy tensors
    completed_stories = test_story_completion_batch(model, partial_stories, imgs=None, max_length=1000)

    # Print results
    for i, (partial, complete) in enumerate(zip(partial_stories, completed_stories)):
        print(f"Partial Story {i+1}: {partial}")
        print(f"Completed Story {i+1}: {complete}\n")


# Run the test with multiple datapoints
run_story_completion_test_batch(model)


In [ ]:
def pil_to_tensor_test(pil_image, img_size=224):
    # Ensure image is in RGB format
    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')

    # Define preprocessing transforms
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),  # Resize to desired dimensions
        transforms.ToTensor(),                   # Convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
    ])

    return transform(pil_image)  # Add batch dimension

In [ ]:
# # Plot both graphs
plot_training_loss(train_losses, epochs)
plot_validation_loss(val_losses, epochs)

In [ ]:
# # Plot both graphs
plot_training_loss(train_losses, epochs)
plot_validation_loss(val_losses, epochs)

In [ ]:
def test_story_completion_batch(model, partial_stories, imgs=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model for multiple datapoints.

    Parameters:
    - model: The trained model.
    - partial_stories: A list of input texts to start the story completion.
    - imgs: A list of image file paths or None for each story. Provide a dummy image if None.
    - max_length: Maximum length of the completed story.
    - img_size: Size to which the image should be resized.
    """
    model.eval()

    completed_stories = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Iterate through the batch of stories
    for i, partial_story in enumerate(partial_stories):
        # Encode the partial story
        encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
        generated_indices = encoded_partial

        # Prepare the image tensor
        if imgs is not None and imgs[i] is not None:
            img_tensor = jpg_to_tensor(imgs[i], img_size).unsqueeze(0).to(device)
        else:
            # Create a dummy image tensor (e.g., all zeros with appropriate shape)
            img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

        # Generate the story
        for _ in range(max_length - len(partial_story)):
            # Forward pass through the model
            logits = model(img_tensor, generated_indices, None)  # Model's forward pass
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get the most likely next token

            # Append the next token to the generated indices
            generated_indices = torch.cat([generated_indices, next_token], dim=1)

            # Stop generating if a padding character or an unwanted token appears
            if next_token.item() == stoi['<pad>']:
                break

        # Decode the generated story
        completed_story = decode(generated_indices[0].tolist())
        completed_stories.append(completed_story)

    return completed_stories


# Function to test story completion with multiple datapoints
def run_story_completion_test_batch(model):
    # Example batch of partial stories
    partial_stories = [
        "I can’t believe my eyes! This place looks magical, like something out of a fairytale! ",
        "The room was dimly lit, yet every corner sparkled with a mysterious glow. ",
        "As I stepped inside, the aroma of fresh flowers overwhelmed my senses. ",
        "The walls were lined with bookshelves, each filled with volumes that seemed ancient and wise. ",
        "Through the open window, I could hear the distant sound of waves crashing on the shore. ",
        "The fireplace crackled gently, casting a warm light across the room. ",
        "Outside, the snow fell silently, covering the world in a blanket of white. ",
        "The garden beyond the glass doors looked like a dream, alive with colors and fragrance. ",
        "In the center of the room, a grand piano stood, inviting me to play a tune. "
    ]

    # Example: No images provided, use dummy tensors
    completed_stories = test_story_completion_batch(model, partial_stories, imgs=None, max_length=1000)

    # Print results
    for i, (partial, complete) in enumerate(zip(partial_stories, completed_stories)):
        print(f"Partial Story {i+1}: {partial}")
        print(f"Completed Story {i+1}: {complete}\n")


# Run the test with multiple datapoints
run_story_completion_test_batch(model)


In [ ]:
from google.colab import files
import shutil

# Original file path of the .pth model on Colab
original_path = '/content/latest_model.pth'  # Replace with your model's actual path

# New file name for the model
renamed_path = '/content/NanoVLMlarge_ShortDesc_25M.pth'

# Rename the file
shutil.move(original_path, renamed_path)


# Download the renamed file
files.download(renamed_path)


#himanshu remaining

In [ ]:
def test_story_completion_batch(model, partial_stories, imgs=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model for multiple datapoints.

    Parameters:
    - model: The trained model.
    - partial_stories: A list of input texts to start the story completion.
    - imgs: A list of image file paths or None for each story. Provide a dummy image if None.
    - max_length: Maximum length of the completed story.
    - img_size: Size to which the image should be resized.
    """
    model.eval()

    completed_stories = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Iterate through the batch of stories
    for i, partial_story in enumerate(partial_stories):
        # Encode the partial story
        encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
        generated_indices = encoded_partial

        # Prepare the image tensor
        if imgs is not None and imgs[i] is not None:
            img_tensor = jpg_to_tensor(imgs[i], img_size).unsqueeze(0).to(device)
        else:
            # Create a dummy image tensor (e.g., all zeros with appropriate shape)
            img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

        # Generate the story
        for _ in range(max_length - len(partial_story)):
            # Forward pass through the model
            logits = model(img_tensor, generated_indices, None)  # Model's forward pass
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get the most likely next token

            # Append the next token to the generated indices
            generated_indices = torch.cat([generated_indices, next_token], dim=1)

            # Stop generating if a padding character or an unwanted token appears
            if next_token.item() == stoi['<pad>']:
                break

        # Decode the generated story
        completed_story = decode(generated_indices[0].tolist())
        completed_stories.append(completed_story)

    return completed_stories


# Function to test story completion with multiple datapoints
def run_story_completion_test_batch(model):
    # Example batch of partial stories
    partial_stories = [
        "I can’t believe my eyes! This place  ",
        "The room was dimly lit, yet every ",
        "As I stepped inside, the aroma  ",
        "The walls were lined with  ",
        "Through the open window,  ",
        "The fireplace crackled gently,  ",
        "Outside, the snow fell silently,  ",
        "The garden beyond the glass  ",
        "In the center of the room,  "
    ]

    # Example: No images provided, use dummy tensors
    completed_stories = test_story_completion_batch(model, partial_stories, imgs=None, max_length=1000)

    # Print results
    for i, (partial, complete) in enumerate(zip(partial_stories, completed_stories)):
        print(f"Partial Story {i+1}: {partial}")
        print(f"Completed Story {i+1}: {complete}\n")


# Run the test with multiple datapoints
run_story_completion_test_batch(model)


In [ ]:
def test_story_completion_batch(model, partial_stories, imgs=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model for multiple datapoints.

    Parameters:
    - model: The trained model.
    - partial_stories: A list of input texts to start the story completion.
    - imgs: A list of image file paths or None for each story. Provide a dummy image if None.
    - max_length: Maximum length of the completed story.
    - img_size: Size to which the image should be resized.
    """
    model.eval()

    completed_stories = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Iterate through the batch of stories
    for i, partial_story in enumerate(partial_stories):
        # Encode the partial story
        encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
        generated_indices = encoded_partial

        # Prepare the image tensor
        if imgs is not None and imgs[i] is not None:
            img_tensor = jpg_to_tensor(imgs[i], img_size).unsqueeze(0).to(device)
        else:
            # Create a dummy image tensor (e.g., all zeros with appropriate shape)
            img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

        # Generate the story
        for _ in range(max_length - len(partial_story)):
            # Forward pass through the model
            logits = model(img_tensor, generated_indices, None)  # Model's forward pass
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get the most likely next token

            # Append the next token to the generated indices
            generated_indices = torch.cat([generated_indices, next_token], dim=1)

            # Stop generating if a padding character or an unwanted token appears
            if next_token.item() == stoi['<pad>']:
                break

        # Decode the generated story
        completed_story = decode(generated_indices[0].tolist())
        completed_stories.append(completed_story)

    return completed_stories


# Function to test story completion with multiple datapoints
def run_story_completion_test_batch(model):
    # Example batch of partial stories
    partial_stories = [
    "The golden sun was shining ",
    "I walked into the room and  ",
    "The air smelled like cookies  ",
    "The tiny birds outside the  ",
    "There was a little cat sleeping  ",
    "Through the glass door, ",
    "The room was quiet, but I  ",
    "The sky outside was painted with ",
    "The soft light of the lamp made  ",
    "The wooden floor creaked  ",
    "The small swing in the ",
    "The table had a vase with fresh  ",
    "There was a little dog wagging ",
    "The sound of the waves crashing  ",
    "The big tree outside was full of  ",
    "The stars were starting to ",
    "The tiny sparkles on the  ",
    "The old rocking chair near  ",
    "The little garden path was  ",
    "The clouds in the sky were so ",
    "The soft breeze coming through the  ",
    "The soft cushions on the sofa made ",
    "The tiny butterflies in the garden",
    "The room smelled like fresh ",
    "The big wooden table had a  "
]



    # Example: No images provided, use dummy tensors
    completed_stories = test_story_completion_batch(model, partial_stories, imgs=None, max_length=1000)

    # Print results
    for i, (partial, complete) in enumerate(zip(partial_stories, completed_stories)):
        print(f"Partial Story {i+1}: {partial}")
        print(f"Completed Story {i+1}: {complete}\n")


# Run the test with multiple datapoints
run_story_completion_test_batch(model)


In [ ]:
def test_story_completion_batch(model, partial_stories, imgs=None, max_length=100, img_size=224):
    """
    Test the story completion task using a trained model for multiple datapoints.

    Parameters:
    - model: The trained model.
    - partial_stories: A list of input texts to start the story completion.
    - imgs: A list of image file paths or None for each story. Provide a dummy image if None.
    - max_length: Maximum length of the completed story.
    - img_size: Size to which the image should be resized.
    """
    model.eval()

    completed_stories = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Iterate through the batch of stories
    for i, partial_story in enumerate(partial_stories):
        # Encode the partial story
        encoded_partial = torch.tensor([[stoi[ch] for ch in partial_story]], dtype=torch.long).to(device)
        generated_indices = encoded_partial

        # Prepare the image tensor
        if imgs is not None and imgs[i] is not None:
            img_tensor = jpg_to_tensor(imgs[i], img_size).unsqueeze(0).to(device)
        else:
            # Create a dummy image tensor (e.g., all zeros with appropriate shape)
            img_tensor = torch.zeros(1, 3, img_size, img_size).to(device)

        # Generate the story
        for _ in range(max_length - len(partial_story)):
            # Forward pass through the model
            logits = model(img_tensor, generated_indices, None)  # Model's forward pass
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # Get the most likely next token

            # Append the next token to the generated indices
            generated_indices = torch.cat([generated_indices, next_token], dim=1)

            # Stop generating if a padding character or an unwanted token appears
            if next_token.item() == stoi['<pad>']:
                break

        # Decode the generated story
        completed_story = decode(generated_indices[0].tolist())
        completed_stories.append(completed_story)

    return completed_stories


# Function to test story completion with multiple datapoints
def run_story_completion_test_batch(model):
    # Example batch of partial stories
    partial_stories = [
    "The gentle wind blew through the trees,  ",
    "The warm sunlight streamed through the  ",
    "I could see tiny birds hopping  ",
    "The small pond outside sparkled  ",
    "There was a big teddy bear sitting ",
    "The sweet smell of chocolate drifted  ",
    "The colorful kites flying in the  ",
    "The soft blanket on the couch ",
    "The big clock on the wall made  ",
    "Through the open window,  ",
    "The tall glass of lemonade ",
    "The shiny wooden staircase  ",
    "The soft breeze brought the scent  ",
    "The moonlight spilled through the ",
    "The wooden chest in the corner ",
    "The little puppy curled up on  ",
    "The sound of crickets outside ",
    "The garden gate creaked open slowly, ",
    "The shiny glass jar on the  ",
    "The gentle ripples on the  ",
    "The swing under the big tree  ",
    "The soft carpet under my feet ",
    "The tiny sparrow sitting on the  ",
    "The stack of storybooks on the  ",
    "The colorful lanterns hanging in "
]




    # Example: No images provided, use dummy tensors
    completed_stories = test_story_completion_batch(model, partial_stories, imgs=None, max_length=1000)

    # Print results
    for i, (partial, complete) in enumerate(zip(partial_stories, completed_stories)):
        print(f"Partial Story {i+1}: {partial}")
        print(f"Completed Story {i+1}: {complete}\n")


# Run the test with multiple datapoints
run_story_completion_test_batch(model)


In [ ]:
from google.colab import files
import shutil

# Original file path of the .pth model on Colab
original_path = '/content/latest_model.pth'  # Replace with your model's actual path

# New file name for the model
renamed_path = '/content/NanoVLMlarge_ShortDesc_25M.pth'

# Rename the file
shutil.move(original_path, renamed_path)


# Download the renamed file
files.download(renamed_path)
